In [ ]:
import pyxdf
import numpy as np
import os
import logging
from PIL import Image


# # for the tests
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.widgets import SpanSelector

# from scipy.signal import butter, filtfilt, find_peaks


# and set to true for testing (default) but can be already set to false from outside (e.g., by the test script)
if "doRunTests" not in globals():
    doRunTests = True

if doRunTests:
    import matplotlib.pyplot as plt
    # the best for debug-test plots (external window that you can make fullscreen and zoom)
    %matplotlib qt


# Compute panu for one xdf file

In [ ]:
def save_panu(xdf_fullFname):
    fname_xdf = os.path.basename(xdf_fullFname)
    fname_panu = fname_xdf.replace(".xdf", "_xdf_panu.csv")

    # load the xdf file

    logging.info(f"Saved '{fname_panu}'")

# Compute panu for one visit

In [ ]:
def get_xdf_files_in_visit(visit_dir, directories_to_skip=None):
    """Get the xdf files in the visit_dir"""

    xdf_files = []

    if not os.path.exists(visit_dir):
        raise ValueError(f"Directory {visit_dir} does not exist")

    for root, dirs, files in os.walk(visit_dir):
        # Skip the directories that are in the directories_to_skip list
        if directories_to_skip and any(
            skip_dir in root for skip_dir in directories_to_skip
        ):
            continue
        for file in files:
            if directories_to_skip and any(
                skip_dir in root for skip_dir in directories_to_skip
            ):
                continue
            if file.endswith(".xdf"):
                xdf_files.append(os.path.join(root, file))

    if not xdf_files:
        logging.warning(f"No xdf files found in {visit_dir}")

    return xdf_files


def is_already_done_panu_in_visit(visitPath, checkLog_fname):
    """
    Check if the visit was already processed with panu
    """
    absVisitPath = os.path.abspath(visitPath)
    full_checkLog_fname = os.path.join(absVisitPath, checkLog_fname)
    return os.path.isfile(full_checkLog_fname)


def create_panu_log_file(visitPath, checkLog_fname):
    """
    Create the log file for panu
    """
    absVisitPath = os.path.abspath(visitPath)
    full_checkLog_fname = os.path.join(absVisitPath, checkLog_fname)

    # Create the log file
    logging.basicConfig(
        filename=full_checkLog_fname,
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        force=True,  # remove previous handlers and set the new one
    )

    return full_checkLog_fname


def merge_panu_png_files_to_pdf(visit_path):
    """
    Merge the panu png files in the visit folder
    """
    png_files = [f for f in os.listdir(visit_path) if f.endswith("_panu.png")]
    png_files.sort()

    if len(png_files) > 1:
        # read the png files
        images = [
            Image.open(os.path.join(visit_path, png_file)) for png_file in png_files
        ]
        # convert to RGB
        images = [img.convert("RGB") for img in images]

        images[0].save(
            os.path.join(visit_path, f"{os.path.basename(visit_path)}_panu_png.pdf"),
            save_all=True,
            append_images=images[1:],
        )

        # remove the original png files
        for png_file in png_files:
            os.remove(os.path.join(visit_path, png_file))
            # print(f"    Removed {png_file}")
    else:
        msg = "No panu png files to merge"
        logging.info(msg)
        print(msg)


def merge_panu_pdf_files_to_pdf(visit_path):
    """
    Merge the panu pdf files in the visit folder
    """
    pdf_files = [f for f in os.listdir(visit_path) if f.endswith("_panu.pdf")]
    pdf_files.sort()

    from pypdf import PdfWriter

    if len(pdf_files) > 1:
        # read the pdf files
        pdf_merger = PdfWriter()
        for pdf_file in pdf_files:
            pdf_merger.append(os.path.join(visit_path, pdf_file))

        pdf_merger.write(
            os.path.join(visit_path, f"{os.path.basename(visit_path)}_panu_pdf.pdf")
        )
        pdf_merger.close()

        # remove the original pdf files
        for pdf_file in pdf_files:
            os.remove(os.path.join(visit_path, pdf_file))
            # print(f"    Removed {pdf_file}")
            pass
    else:
        msg = "No panu pdf files to merge"
        logging.info(msg)
        print(msg)


def get_panu_in_visit(visit_dir, directories_to_skip=None):
    """Correct the kinect timestamps for all the xdf files in the visit_dir"""

    panu_log = "panu.log"
    xdf_files = get_xdf_files_in_visit(visit_dir, directories_to_skip)

    if not xdf_files or len(xdf_files) == 0:
        return

    if is_already_done_panu_in_visit(visit_dir, panu_log):
        print(f"    Already done: '{panu_log}' found")
        return

    create_panu_log_file(visit_dir, panu_log)
    logging.info(f"Starting panu in {visit_dir}")

    for xdf_fullFname in xdf_files:
        print(f"---- \n{xdf_fullFname}")
        logging.info(f"{os.path.basename(xdf_fullFname)}")
        save_panu(xdf_fullFname)

    merge_panu_png_files_to_pdf(os.path.dirname(visit_dir))
    logging.info("panu completed")
    print(f"    panu completed: see '{panu_log}' for details")


if doRunTests:
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1"
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2"
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3"
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210716_V1"
    # visit_dir = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210820_V2"
    # visit_dir = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20211116_V3"

    visit_dir = "../dat/ReArm.lnk/C1P42/V1"
    visit_dir = "../dat/ReArm.lnk/C1P42/V2"
    visit_dir = "../dat/ReArm.lnk/C1P42/V3"

    visit_dir = (
        "../dat/ReArm.lnk/DATA_named/C1P02/V2"  # /Armeo/002_CorJea_20210409_2_a.xdf"
    )

    visit_dir = "../dat/ReArm.lnk/DATA_named/C1P01/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P21/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P23/V1"
    visit_dir = "../dat/ReArm.lnk/DATA_named/C1P20/V2"

    visit_dir = "../dat/ReArm.lnk/DATA_named/C1P38/V1"

    os.remove(os.path.join(visit_dir, "panu.log"))
    get_panu_in_visit(visit_dir, directories_to_skip=["old", "Training", "Armeo"])